# N03: SigLIP, searching the image itself

A different branch: no text about the painting at all. SigLIP was trained on image-caption pairs so that a sentence
and a matching image land close together; we compare the visitor's query (text tower) with each painting (image
tower). R4 is compared with the baseline R1, not with R3 (it changes the encoder, the modality and the text at once).

The image embeddings are computed once and cached; at query time only the query goes through the text tower.

In [ ]:
import pandas as pd

from deep_learning_project_waerniers_lamy.nlp import utils as u

pd.set_option("display.width", 160)
from pathlib import Path

In [ ]:
paintings = u.load_corpus("dev")
descriptions = u.load_descriptions()
queries = u.QUERIES["text"].tolist()
scores = {
    "R0 random": u.rung_random(paintings, queries),
    "R1 metadata": u.rung_metadata(paintings, queries),
    "R2 description": u.rung_description(paintings, queries, descriptions),
    "R3 description+metadata": u.rung_description_metadata(paintings, queries, descriptions),
    "R4 siglip image": u.rung_siglip(paintings, queries),
}
u.write_judging_page("dev", scores, paintings)

### Judging (by hand)

Open the page printed above in a browser. It shows **only the query and the image** (no rung name, artist, style or
description), and only paintings not judged before. Grade each one 0 / 1 / 2 with the keyboard, click *export JSON*,
then run the next cell. Grades are kept in the browser, so you can stop and come back.

In [ ]:
u.import_judgments("dev", Path.home() / "Downloads" / "judgments_dev.json")

## Results

Pre-registered comparison 2 of 3: **does a visual embedding beat metadata?** (R4 vs R1).

In [ ]:
per_query = u.per_query_scores("dev", scores, paintings)
table = u.summary_table(per_query)
u.plot_ladder(table, "dev: + SigLIP")
display(table)
u.compare(per_query, [("R4 siglip image", "R1 metadata")])

In [ ]:
per_query.groupby(["rung", "category"])["ndcg@5"].mean().unstack().round(3)